# Detection Sandbox

Interactive notebook for testing and tuning face/eye detection parameters.

In [ ]:
import sys
import os
sys.path.append(os.path.dirname(os.path.dirname(os.path.abspath('.'))))

import cv2
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, clear_output
import time

from src.detection import load_cascades, detect_face_and_eyes, draw_detections
import config

%matplotlib inline

## Test Webcam Access

In [ ]:
# Quick webcam test
cap = cv2.VideoCapture(config.CAMERA_INDEX)
ret, frame = cap.read()
cap.release()

if ret:
    print(f"✓ Webcam working. Frame shape: {frame.shape}")
    # Display the frame
    plt.figure(figsize=(8, 6))
    plt.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    plt.title("Webcam Test Frame")
    plt.axis('off')
    plt.show()
else:
    print("Error: Cannot read from webcam")

## Load Cascade Classifiers

In [ ]:
face_cascade, eye_cascade = load_cascades()
print("Cascades loaded successfully")

## Capture and Analyze Single Frame

In [ ]:
def capture_and_detect():
    """Capture single frame and run detection."""
    cap = cv2.VideoCapture(config.CAMERA_INDEX)
    cap.set(cv2.CAP_PROP_FRAME_WIDTH, config.STANDARD_FRAME_WIDTH)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, config.STANDARD_FRAME_HEIGHT)
    
    # Wait a moment for camera to stabilize
    for _ in range(5):
        cap.read()
    
    ret, frame = cap.read()
    cap.release()
    
    if not ret:
        print("Failed to capture frame")
        return None, None
    
    # Run detection
    detections = detect_face_and_eyes(frame, face_cascade, eye_cascade)
    
    return frame, detections

# Capture and analyze
frame, detections = capture_and_detect()

if frame is not None:
    # Determine state
    if not detections:
        state = "Absent"
    else:
        face_box, eyes = detections[0]
        if len(eyes) == 0:
            state = "Drowsy"
        elif len(eyes) >= 2:
            state = "Focused"
        else:
            state = "Distracted"
    
    # Draw detections
    display_frame = draw_detections(frame, detections, state)
    
    # Show result
    plt.figure(figsize=(12, 8))
    plt.imshow(cv2.cvtColor(display_frame, cv2.COLOR_BGR2RGB))
    plt.title(f"Detection Result - State: {state}")
    plt.axis('off')
    plt.show()
    
    # Print detection details
    print(f"\nDetection Summary:")
    print(f"- Faces detected: {len(detections)}")
    if detections:
        face_box, eyes = detections[0]
        print(f"- Face box: x={face_box[0]}, y={face_box[1]}, w={face_box[2]}, h={face_box[3]}")
        print(f"- Eyes detected: {len(eyes)}")
        print(f"- Determined state: {state}")

## Test Detection Parameters

In [ ]:
def test_parameters(scale_factor, min_neighbors):
    """Test detection with different parameters."""
    cap = cv2.VideoCapture(config.CAMERA_INDEX)
    ret, frame = cap.read()
    cap.release()
    
    if not ret:
        return None
    
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    
    # Test face detection with custom parameters
    faces = face_cascade.detectMultiScale(
        gray,
        scaleFactor=scale_factor,
        minNeighbors=min_neighbors,
        minSize=config.FACE_MIN_SIZE
    )
    
    return len(faces)

# Test different parameter combinations
print("Testing different detection parameters:")
print("(scaleFactor, minNeighbors) -> faces detected\n")

for sf in [1.05, 1.1, 1.2]:
    for mn in [3, 5, 7]:
        faces = test_parameters(sf, mn)
        if faces is not None:
            print(f"({sf:.2f}, {mn}) -> {faces} faces")

## Compare Eye Cascade Options

In [ ]:
# Compare different eye cascades
cascade_path = cv2.data.haarcascades

eye_cascades = {
    'haarcascade_eye.xml': cv2.CascadeClassifier(cascade_path + 'haarcascade_eye.xml'),
    'haarcascade_eye_tree_eyeglasses.xml': cv2.CascadeClassifier(cascade_path + 'haarcascade_eye_tree_eyeglasses.xml')
}

# Capture frame
cap = cv2.VideoCapture(config.CAMERA_INDEX)
ret, frame = cap.read()
cap.release()

if ret:
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    
    # Detect face first
    faces = face_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5)
    
    if len(faces) > 0:
        x, y, w, h = faces[0]
        face_roi = gray[y:y + h, x:x + w]
        upper_face = face_roi[0:int(h * 0.6), :]
        
        print("Eye cascade comparison:")
        for name, cascade in eye_cascades.items():
            eyes = cascade.detectMultiScale(upper_face, scaleFactor=1.1, minNeighbors=6)
            print(f"- {name}: {len(eyes)} eyes detected")
    else:
        print("No face detected for eye cascade comparison")

## Test Detection Behaviors

In [ ]:
print("Detection Behavior Test")
print("="*40)
print("\nPlease perform the following actions when prompted:")
print("1. Look straight at camera with eyes open")
print("2. Close your eyes")
print("3. Turn your head to the side")
print("4. Look down or sideways\n")

actions = [
    "Look straight with eyes open",
    "Close your eyes",
    "Turn head to the side",
    "Look down or sideways"
]

results = []

for action in actions:
    input(f"\nPress Enter when ready to: {action}")
    time.sleep(1)  # Give time to position
    
    frame, detections = capture_and_detect()
    
    if frame is not None:
        if not detections:
            state = "Absent"
            eyes_count = 0
        else:
            face_box, eyes = detections[0]
            eyes_count = len(eyes)
            if eyes_count == 0:
                state = "Drowsy"
            elif eyes_count >= 2:
                state = "Focused"
            else:
                state = "Distracted"
        
        results.append((action, state, eyes_count))
        print(f"Result: State={state}, Eyes={eyes_count}")

# Summary
print("\n" + "="*40)
print("Summary of Detection Behaviors:")
print("="*40)
for action, state, eyes in results:
    print(f"{action:30} -> State: {state:10} (Eyes: {eyes})")